# Run the Staging Component

A Jupyter notebook that runs a staging component, validates readiness, executes smoke tests, and captures diagnostics.

## 1. Section: Set Up Environment Variables and Configuration

Load environment variables from local env files and the notebook session, define staging-specific settings, and print a redacted summary.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path


def load_env_file(path: Path) -> int:
    loaded = 0
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)
            loaded += 1
    return loaded


loaded_sources = []
for env_name in [".env.staging", ".env.local", ".env"]:
    env_path = Path(env_name)
    if env_path.exists():
        loaded_count = load_env_file(env_path)
        loaded_sources.append({"file": str(env_path), "keys_loaded": loaded_count})

# Staging defaults for this repo's Vite staging page.
os.environ.setdefault("VITE_APP_ENV", "staging")
os.environ.setdefault("VITE_ENABLE_STAGING_MODEL_TEST", "1")
os.environ.setdefault("VITE_STAGING_REQUEST_TIMEOUT_MS", "300000")
os.environ.setdefault("STAGING_HOST", "127.0.0.1")
os.environ.setdefault("STAGING_PORT", "3000")

staging_config = {
    "host": os.getenv("STAGING_HOST", "127.0.0.1"),
    "port": int(os.getenv("STAGING_PORT", "3000")),
    "base_url": f"http://{os.getenv('STAGING_HOST', '127.0.0.1')}:{int(os.getenv('STAGING_PORT', '3000'))}",
    "app_env": os.getenv("VITE_APP_ENV", "staging"),
    "enable_model_test": os.getenv("VITE_ENABLE_STAGING_MODEL_TEST", "1"),
    "request_timeout_ms": int(os.getenv("VITE_STAGING_REQUEST_TIMEOUT_MS", "300000")),
    "hf_api_key": os.getenv("VITE_HF_API_KEY", ""),
    "vllm_base_url": os.getenv("VITE_VLLM_BASE_URL", ""),
    "vllm_api_key": os.getenv("VITE_VLLM_API_KEY", ""),
    "captured_at": datetime.utcnow().isoformat() + "Z",
}


def redact(secret: str) -> str:
    if not secret:
        return ""
    if len(secret) <= 8:
        return "***"
    return secret[:3] + "..." + secret[-3:]


redacted_summary = {
    **staging_config,
    "hf_api_key": redact(staging_config["hf_api_key"]),
    "vllm_api_key": redact(staging_config["vllm_api_key"]),
}

print("Loaded env sources:")
print(json.dumps(loaded_sources, indent=2))
print("\nRedacted staging config:")
print(json.dumps(redacted_summary, indent=2))

## 2. Section: Install Dependencies and Verify Tooling

Install or verify required packages and CLIs, then run version checks to confirm the runtime and orchestration tooling are available.

In [ ]:
import shutil
import subprocess
import sys


def run_command(cmd):
    proc = subprocess.run(cmd, text=True, capture_output=True)
    output = (proc.stdout or proc.stderr).strip()
    return {
        "command": " ".join(cmd),
        "returncode": proc.returncode,
        "output": output.splitlines()[:4],
    }


def ensure_python_package(package_name: str):
    try:
        __import__(package_name)
        return {"package": package_name, "installed": True, "action": "already_available"}
    except Exception:
        install = subprocess.run(
            [sys.executable, "-m", "pip", "install", package_name],
            text=True,
            capture_output=True,
        )
        return {
            "package": package_name,
            "installed": install.returncode == 0,
            "action": "installed" if install.returncode == 0 else "install_failed",
        }


package_checks = [
    ensure_python_package("requests"),
]

cli_checks = {
    "python": run_command([sys.executable, "--version"]),
    "pip": run_command([sys.executable, "-m", "pip", "--version"]),
    "node": run_command(["node", "--version"]) if shutil.which("node") else {"missing": True},
    "npm": run_command(["npm", "--version"]) if shutil.which("npm") else {"missing": True},
    "docker": run_command(["docker", "--version"]) if shutil.which("docker") else {"missing": True},
    "docker_compose": run_command(["docker", "compose", "version"]) if shutil.which("docker") else {"missing": True},
}

print("Package checks:")
print(json.dumps(package_checks, indent=2))
print("\nCLI checks:")
print(json.dumps(cli_checks, indent=2))

## 3. Section: Start Required Local Services

Start local dependencies (for example via Docker Compose) and poll until each reports a healthy or running state.

In [ ]:
import time

compose_started = False
dependency_status = []

def run_process(cmd):
    return subprocess.run(cmd, text=True, capture_output=True)

compose_file_present = Path("docker-compose.yml").exists()
docker_available = shutil.which("docker") is not None

if compose_file_present and docker_available:
    up = run_process(["docker", "compose", "up", "-d"])
    if up.returncode == 0:
        compose_started = True

    services_proc = run_process(["docker", "compose", "config", "--services"])
    services = [line.strip() for line in services_proc.stdout.splitlines() if line.strip()] if services_proc.returncode == 0 else []

    timeout_seconds = 180
    poll_interval = 3
    start = time.time()

    while time.time() - start < timeout_seconds:
        ps_json = run_process(["docker", "compose", "ps", "--format", "json"])
        observed = {}

        if ps_json.returncode == 0 and ps_json.stdout.strip():
            for line in ps_json.stdout.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                except Exception:
                    continue
                name = str(record.get("Service") or record.get("Name") or "").strip()
                state = str(record.get("State") or record.get("Status") or "").strip()
                health = str(record.get("Health") or "").strip().lower()
                if name:
                    observed[name] = {"state": state, "health": health}

        dependency_status = []
        all_ready = True
        for svc in services:
            rec = observed.get(svc, {"state": "", "health": ""})
            state = rec.get("state", "")
            health = rec.get("health", "")
            ready = ("running" in state.lower() or "up" in state.lower()) and (health in {"", "healthy"})
            dependency_status.append({"service": svc, "state": state, "health": health, "ready": ready})
            if not ready:
                all_ready = False

        if services and all_ready:
            break
        if not services:
            break
        time.sleep(poll_interval)
else:
    dependency_status = [{
        "service": "compose",
        "state": "skipped",
        "health": "n/a",
        "ready": True,
        "reason": "docker-compose.yml not found or Docker unavailable",
    }]

print(json.dumps({
    "compose_file_present": compose_file_present,
    "docker_available": docker_available,
    "compose_started": compose_started,
    "dependency_status": dependency_status,
}, indent=2))

## 4. Section: Run the Staging Component

Start the staging component process, stream startup logs in the notebook, and capture process metadata (PID, start time, and detected port).

In [ ]:
import re
import threading

component_proc = None
component_output_thread = None
component_log_lines = []
component_metadata = {}


def stream_component_output(proc):
    while True:
        line = proc.stdout.readline()
        if not line:
            break
        text = line.rstrip("\n")
        component_log_lines.append(text)
        print(text)


startup_env = os.environ.copy()
startup_env["VITE_APP_ENV"] = startup_env.get("VITE_APP_ENV", "staging")
startup_env["VITE_ENABLE_STAGING_MODEL_TEST"] = startup_env.get("VITE_ENABLE_STAGING_MODEL_TEST", "1")

command = ["npm", "run", "dev"]
start_ts = datetime.utcnow().isoformat() + "Z"

component_proc = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=startup_env,
)

component_output_thread = threading.Thread(target=stream_component_output, args=(component_proc,), daemon=True)
component_output_thread.start()

port = int(staging_config.get("port", 3000))
ready_deadline = time.time() + 30

while time.time() < ready_deadline:
    if component_proc.poll() is not None:
        break
    for line in component_log_lines[-20:]:
        match = re.search(r"http://localhost:(\d+)/", line)
        if match:
            port = int(match.group(1))
    if any("ready in" in line.lower() for line in component_log_lines[-20:]):
        break
    time.sleep(0.5)

staging_config["port"] = port
staging_config["base_url"] = f"http://{staging_config['host']}:{port}"

component_metadata = {
    "pid": component_proc.pid,
    "start_time_utc": start_ts,
    "command": command,
    "bound_port": port,
    "base_url": staging_config["base_url"],
    "process_running": component_proc.poll() is None,
}

print("\nComponent metadata:")
print(json.dumps(component_metadata, indent=2))

## 5. Section: Validate Health and Readiness Endpoints

Query health/readiness URLs with retry and backoff, assert expected status codes, and verify response content before proceeding.

In [ ]:
import urllib.error
import urllib.request

readiness_results = []


def http_get(url: str, timeout: int = 8):
    req = urllib.request.Request(url, method="GET")
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = resp.read().decode("utf-8", errors="replace")
        return resp.getcode(), body


endpoints = [
    {
        "name": "health",
        "url": f"{staging_config['base_url']}/",
        "expected_status": 200,
        "expected_contains": ["MathPulse"],
    },
    {
        "name": "readiness",
        "url": f"{staging_config['base_url']}/staging/model-test",
        "expected_status": 200,
        "expected_contains": ["STAGING", "Model"],
    },
]

max_attempts = 8
backoff_seconds = 1
all_ready = False

for attempt in range(1, max_attempts + 1):
    readiness_results = []
    all_ready = True

    for ep in endpoints:
        result = {
            "name": ep["name"],
            "url": ep["url"],
            "attempt": attempt,
            "ok": False,
            "status": None,
            "contains_checks": {},
            "error": "",
        }
        try:
            status, body = http_get(ep["url"])
            result["status"] = status
            status_ok = status == ep["expected_status"]
            contains_checks = {needle: (needle.lower() in body.lower()) for needle in ep["expected_contains"]}
            content_ok = all(contains_checks.values())
            result["contains_checks"] = contains_checks
            result["ok"] = status_ok and content_ok
            if not result["ok"]:
                result["body_snippet"] = body[:400]
        except Exception as exc:
            result["error"] = str(exc)
            result["ok"] = False

        if not result["ok"]:
            all_ready = False
        readiness_results.append(result)

    if all_ready:
        break
    time.sleep(backoff_seconds)
    backoff_seconds = min(backoff_seconds * 2, 10)

print(json.dumps({
    "all_ready": all_ready,
    "attempts_used": max(r.get("attempt", 0) for r in readiness_results) if readiness_results else 0,
    "results": readiness_results,
}, indent=2))

if not all_ready:
    raise RuntimeError("Staging component failed health/readiness validation before timeout.")

## 6. Section: Execute Smoke Tests Against Staging

Run focused HTTP smoke tests against staging endpoints, summarize pass/fail metrics, and show failed-case payload snippets.

In [ ]:
smoke_plan = [
    {
        "name": "root_page",
        "url": f"{staging_config['base_url']}/",
        "expected_status": 200,
        "must_contain": ["MathPulse"],
    },
    {
        "name": "staging_component_page",
        "url": f"{staging_config['base_url']}/staging/model-test",
        "expected_status": 200,
        "must_contain": ["STAGING", "Comparison"],
    },
]

smoke_results = []
for test_case in smoke_plan:
    result = {
        "name": test_case["name"],
        "url": test_case["url"],
        "status": None,
        "passed": False,
        "error": "",
        "body_snippet": "",
    }
    try:
        status, body = http_get(test_case["url"], timeout=10)
        result["status"] = status
        status_ok = status == test_case["expected_status"]
        content_ok = all(token.lower() in body.lower() for token in test_case["must_contain"])
        result["passed"] = status_ok and content_ok
        if not result["passed"]:
            result["body_snippet"] = body[:400]
    except Exception as exc:
        result["error"] = str(exc)
    smoke_results.append(result)

passed_count = sum(1 for r in smoke_results if r["passed"])
failed = [r for r in smoke_results if not r["passed"]]

print(json.dumps({
    "total": len(smoke_results),
    "passed": passed_count,
    "failed": len(smoke_results) - passed_count,
    "results": smoke_results,
}, indent=2))

if failed:
    print("\nFailed smoke test details:")
    for item in failed:
        print(json.dumps(item, indent=2))

## 7. Section: Inspect Logs and Capture Diagnostics

Collect recent app and dependency logs, filter warnings/errors, and export structured diagnostic artifacts for troubleshooting.

In [ ]:
import re

artifact_dir = Path("artifacts") / f"staging_diagnostics_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}"
artifact_dir.mkdir(parents=True, exist_ok=True)

component_logs_text = "\n".join(component_log_lines)
dependency_logs_text = ""
if compose_file_present and docker_available:
    dep_logs = run_process(["docker", "compose", "logs", "--tail", "250"])
    dependency_logs_text = ((dep_logs.stdout or "") + "\n" + (dep_logs.stderr or "")).strip()

error_warning_pattern = re.compile(r"\b(error|warn|warning|fatal|exception)\b", re.IGNORECASE)
app_flagged_lines = [line for line in component_log_lines if error_warning_pattern.search(line)]
dep_flagged_lines = [line for line in dependency_logs_text.splitlines() if error_warning_pattern.search(line)]

env_summary = {
    "captured_at": datetime.utcnow().isoformat() + "Z",
    "staging_config": redacted_summary,
    "component_metadata": component_metadata,
    "dependency_status": dependency_status,
}

(artifact_dir / "environment_summary.json").write_text(json.dumps(env_summary, indent=2), encoding="utf-8")
(artifact_dir / "readiness_results.json").write_text(json.dumps(readiness_results, indent=2), encoding="utf-8")
(artifact_dir / "smoke_results.json").write_text(json.dumps(smoke_results, indent=2), encoding="utf-8")
(artifact_dir / "component_logs.txt").write_text(component_logs_text, encoding="utf-8")
(artifact_dir / "dependency_logs.txt").write_text(dependency_logs_text, encoding="utf-8")
(artifact_dir / "error_warning_summary.json").write_text(
    json.dumps(
        {
            "app_flagged_count": len(app_flagged_lines),
            "dependency_flagged_count": len(dep_flagged_lines),
            "app_flagged_sample": app_flagged_lines[:50],
            "dependency_flagged_sample": dep_flagged_lines[:50],
        },
        indent=2,
    ),
    encoding="utf-8",
)

diagnostics_summary = {
    "artifact_dir": str(artifact_dir),
    "files": sorted(p.name for p in artifact_dir.glob("*")),
    "app_flagged_count": len(app_flagged_lines),
    "dependency_flagged_count": len(dep_flagged_lines),
}

print(json.dumps(diagnostics_summary, indent=2))

## 8. Section: Stop Services and Clean Up Resources

Gracefully stop the staging component and dependencies, then verify no related process remains running.

In [ ]:
cleanup = {
    "component_stopped": False,
    "component_forced_kill": False,
    "compose_stopped": False,
    "process_running_after_cleanup": None,
}

if component_proc is not None and component_proc.poll() is None:
    component_proc.terminate()
    try:
        component_proc.wait(timeout=20)
        cleanup["component_stopped"] = True
    except subprocess.TimeoutExpired:
        component_proc.kill()
        cleanup["component_forced_kill"] = True
        cleanup["component_stopped"] = True

if compose_started and docker_available:
    down = run_process(["docker", "compose", "down", "--remove-orphans"])
    cleanup["compose_stopped"] = down.returncode == 0

cleanup["process_running_after_cleanup"] = component_proc is not None and component_proc.poll() is None

print(json.dumps(cleanup, indent=2))
if cleanup["process_running_after_cleanup"]:
    raise RuntimeError("Staging component process is still running after cleanup.")